In [ ]:
# ==============================================================================
# ⚙️  CONFIG — update these paths before running
# ==============================================================================
BASE_DIR         = "data/hdf5_data_final"
CHECKPOINT_PATH  = "checkpoints/best_checkpoint"


In [28]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import numpy as np
import random
import glob
from tqdm import tqdm

# ==============================================================================
# 📂 CELL 1: SETUP AND SMART PATH DISCOVERY
# ==============================================================================

# Using your exact paths!
BASE_DIR = "data/hdf5_data_final"
CHECKPOINT_PATH = "checkpoints/best_checkpoint"

# 1. Auto-discover the 45 session directories
session_dirs = sorted([
    d for d in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, d)) and not d.startswith('.')
])

session_train_files = []
session_val_files = []
missing_val_sessions = []
test_files = []

print(f"🔎 Found {len(session_dirs)} day adapter sessions. Scanning for train/val/test data...\n")

for d in session_dirs:
    session_path = os.path.join(BASE_DIR, d)
    
    # Flexibly find the train, val, and test data inside each of the 45 session folders
    train_matches = glob.glob(os.path.join(session_path, "*train*"))
    val_matches = glob.glob(os.path.join(session_path, "*val*"))
    test_matches = glob.glob(os.path.join(session_path, "*test*"))
    
    # --- Check Train Data ---
    if not train_matches:
        print(f"⚠️ SKIP: Session {d} has no TRAIN data.")
        continue 
        
    session_train_files.append(train_matches[0])

    # --- Check Val Data ---
    if val_matches:
        session_val_files.append(val_matches[0])
    else:
        # Signal Auto-Split if Val is missing
        session_val_files.append(None) 
        missing_val_sessions.append(d)
        
    # --- Track Test Data ---
    if test_matches:
        test_files.append(test_matches[0])

print(f"🚨 Found {len(missing_val_sessions)} sessions with NO validation file (Auto-Split enabled)")
print(f"✅ Final Lists Ready! Train: {len(session_train_files)} | Val: {len(session_val_files)} | Test: {len(test_files)}")

# Ensure we found data!
assert len(session_train_files) > 0, "❌ Error: Could not find any training files!"
assert len(session_train_files) == len(session_val_files), "❌ Error: Train/Val lists are out of sync!"

print("🎉 Looks perfect! Data successfully loaded.")


🔎 Found 45 day adapter sessions. Scanning for train/val/test data...

🚨 Found 0 sessions with NO validation file (Auto-Split enabled)
✅ Final Lists Ready! Train: 45 | Val: 45 | Test: 41
🎉 Looks perfect! Data successfully loaded.


In [29]:
# Count how many ACTUAL validation files we have (ignoring the None placeholders)
actual_val_count = len([x for x in session_val_files if x is not None])

print(f"✅ Final Lists Ready! Train: {len(session_train_files)} | Val: {actual_val_count} (plus {len(missing_val_sessions)} Auto-Split placeholders) | Test: {len(test_files)}")


✅ Final Lists Ready! Train: 45 | Val: 45 (plus 0 Auto-Split placeholders) | Test: 41


In [30]:
import h5py

def load_h5py_file(file_path):
    data = {
        'neural_features': [],
        'n_time_steps': [],
        'seq_class_ids': [],
        'seq_len': [],
        'transcriptions': [],
        'sentence_label': [],
        'session': [],
        'block_num': [],
        'trial_num': [],
    }
    # Open the hdf5 file for that day
    with h5py.File(file_path, 'r') as f:

        keys = list(f.keys())

        # For each trial in the selected trials in that day
        for key in keys:
            g = f[key]

            neural_features = g['input_features'][:]
            n_time_steps = g.attrs['n_time_steps']
            seq_class_ids = g['seq_class_ids'][:] if 'seq_class_ids' in g else None
            seq_len = g.attrs['seq_len'] if 'seq_len' in g.attrs else None
            transcription = g['transcription'][:] if 'transcription' in g else None
            sentence_label = g.attrs['sentence_label'][:] if 'sentence_label' in g.attrs else None
            session = g.attrs['session']
            block_num = g.attrs['block_num']
            trial_num = g.attrs['trial_num']

            data['neural_features'].append(neural_features)
            data['n_time_steps'].append(n_time_steps)
            data['seq_class_ids'].append(seq_class_ids)
            data['seq_len'].append(seq_len)
            data['transcriptions'].append(transcription)
            data['sentence_label'].append(sentence_label)
            data['session'].append(session)
            data['block_num'].append(block_num)
            data['trial_num'].append(trial_num)
    return data


In [31]:
file_path = 'data/hdf5_data_final/t15.2023.08.13/data_train.hdf5'

data = load_h5py_file(file_path)


In [32]:
print("Number of trials:", len(data['neural_features']))

i = 5  # choose any trial index you want

print("\n=== BASIC INFO ===")
print("Neural shape:", data['neural_features'][i].shape)
print("Time steps:", data['n_time_steps'][i])
print("Phoneme length (seq_len):", data['seq_len'][i])

# -------- Decode transcription (UTF-32 LE) --------
if data['transcriptions'][i] is not None:
    transcription_raw = data['transcriptions'][i]
    transcription = transcription_raw.tobytes().decode("utf-32-le").replace("\x00", "")
else:
    transcription = None

print("\n=== TEXT ===")
print("Transcription (clean text):", transcription)

# -------- Sentence label (UTF-8 bytes) --------
sentence_label = data['sentence_label'][i]
if isinstance(sentence_label, bytes):
    sentence_label = sentence_label.decode("utf-8")

print("Sentence Label:", sentence_label)

# -------- Metadata --------
print("\n=== METADATA ===")
print("Session:", data['session'][i])
print("Block:", data['block_num'][i])
print("Trial:", data['trial_num'][i])   # ✅ fixed bug here

# -------- TARGET: phoneme IDs --------
print("\n=== TRAINING TARGET ===")
seq_ids = data['seq_class_ids'][i]

if seq_ids is not None:
    print("Phoneme class IDs:", seq_ids)
    print("Number of phonemes:", len(seq_ids))
else:
    print("No phoneme labels available for this trial")


Number of trials: 348

=== BASIC INFO ===
Neural shape: (896, 512)
Time steps: 896
Phoneme length (seq_len): 27

=== TEXT ===
Transcription (clean text): I kind of look at it this way.
Sentence Label: I kind of look at it this way.

=== METADATA ===
Session: t15.2023.08.13
Block: 1
Trial: 5

=== TRAINING TARGET ===
Phoneme class IDs: [ 6 40 20  6 23  9 40  3 35 40 21 33 20 40  2 31 40 17 31 40 10 17 29 40
 36 13 40  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0 

In [33]:
import h5py

def load_one_trial(file_path, trial_key):
    """
    Load ONE trial from an HDF5 file
    """

    with h5py.File(file_path, "r") as f:
        g = f[trial_key]

        # -------- neural features --------
        neural_features = g["input_features"][:]     # (T, 512)

        # -------- time steps --------
        n_time_steps = g.attrs["n_time_steps"]       # T

        # -------- phoneme targets --------
        seq_len = g.attrs["seq_len"]                 # true phoneme length
        phoneme_ids = g["seq_class_ids"][:seq_len]   # remove padding

        # -------- metadata --------
        session = g.attrs["session"]
        block_num = g.attrs["block_num"]
        trial_num = g.attrs["trial_num"]

        # decode session if stored as bytes
        if isinstance(session, bytes):
            session = session.decode("utf-8")

    return {
    "neural_features": neural_features,
    "seq_class_ids": phoneme_ids,   # renamed here
    "seq_len": seq_len,
    "n_time_steps": n_time_steps,
    "session": session,
    "block_num": block_num,
    "trial_num": trial_num
}


In [34]:
# Updated path to include 'brain2'
h5_path = "data/hdf5_data_final/t15.2023.08.18/data_train.hdf5"

with h5py.File(h5_path, 'r') as f:
    trial_key = list(f.keys())[0]   # pick first trial

trial = load_one_trial(h5_path, trial_key)

print("Neural shape:", trial["neural_features"].shape)
print("Phoneme IDs:", trial["seq_class_ids"])
print("Phoneme length:", trial["seq_len"])
print("Time steps:", trial["n_time_steps"])
print("Session:", trial["session"])


Neural shape: (486, 512)
Phoneme IDs: [17 31 29 40 15 25 17 24 40 31 34 40 20  3 23 31 17 23 37 34 40]
Phoneme length: 21
Time steps: 486
Session: t15.2023.08.18
